# Smart Sorting Algorithm Performance Analyzer
### Design and Analysis of Algorithms (DAA) Mini Project

**Main feature:** choose an input size and data pattern, click **Recommend Best Algorithm**, and the system benchmarks six sorting algorithms and recommends the fastest one for that specific experiment.

The recommendation is based on measured execution time in the current Colab session. Results can vary between runs and computers, so the winner is not claimed to be universally best.

**Algorithms:** Bubble Sort, Selection Sort, Insertion Sort, Merge Sort, Quick Sort, and Heap Sort.

**Workflow:** user settings → generate input → validate algorithm outputs → measure repeated run times → recommend the fastest observed algorithm → show comparison table and chart.

## 1. Import libraries and prepare Colab widgets

Run the notebook from top to bottom once. No manual `input()` prompts are used; the main project runs from the interactive button below.

In [ ]:
import random
import time
import pandas as pd
import matplotlib.pyplot as plt
try:
    import ipywidgets as widgets
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "ipywidgets"])
    import ipywidgets as widgets
from IPython.display import display, clear_output, FileLink

# Enable ipywidgets in Google Colab when the Colab widget manager is available.
try:
    from google.colab import output as colab_output
    colab_output.enable_custom_widget_manager()
except Exception:
    pass

random.seed(42)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

print("Libraries loaded. Use the interactive controls near the end of this notebook.")

## 2. Generate test datasets

The analyzer supports random, already-sorted, reverse-sorted, and nearly-sorted inputs. For one recommendation, every algorithm receives the same base dataset.

In [ ]:
def generate_data(size, data_type="random"):
    """Return a list of integers with the requested size and input pattern."""
    if not isinstance(size, int) or size < 1:
        raise ValueError("Dataset size must be a positive integer.")

    if data_type == "random":
        return [random.randint(1, 100_000) for _ in range(size)]
    if data_type == "sorted":
        return list(range(size))
    if data_type == "reverse":
        return list(range(size, 0, -1))
    if data_type == "nearly_sorted":
        data = list(range(size))
        swaps = max(1, size // 50)
        for _ in range(swaps):
            i = random.randrange(size)
            j = random.randrange(size)
            data[i], data[j] = data[j], data[i]
        return data

    raise ValueError("Unknown data type. Choose random, sorted, reverse, or nearly_sorted.")

## 3. Implement six sorting algorithms

Each function returns a sorted copy. This keeps the source data unchanged and makes the comparison fair.

In [ ]:
def bubble_sort(arr):
    a = arr.copy()
    n = len(a)
    for i in range(n):
        swapped = False
        for j in range(0, n - i - 1):
            if a[j] > a[j + 1]:
                a[j], a[j + 1] = a[j + 1], a[j]
                swapped = True
        if not swapped:
            break
    return a


def selection_sort(arr):
    a = arr.copy()
    n = len(a)
    for i in range(n):
        min_index = i
        for j in range(i + 1, n):
            if a[j] < a[min_index]:
                min_index = j
        a[i], a[min_index] = a[min_index], a[i]
    return a


def insertion_sort(arr):
    a = arr.copy()
    for i in range(1, len(a)):
        key = a[i]
        j = i - 1
        while j >= 0 and a[j] > key:
            a[j + 1] = a[j]
            j -= 1
        a[j + 1] = key
    return a


def merge_sort(arr):
    if len(arr) <= 1:
        return arr.copy()
    mid = len(arr) // 2
    left = merge_sort(arr[:mid])
    right = merge_sort(arr[mid:])
    merged = []
    i = j = 0
    while i < len(left) and j < len(right):
        if left[i] <= right[j]:
            merged.append(left[i]); i += 1
        else:
            merged.append(right[j]); j += 1
    merged.extend(left[i:])
    merged.extend(right[j:])
    return merged


def quick_sort(arr):
    """Three-way Quick Sort; handles duplicate values without deep duplicate recursion."""
    if len(arr) <= 1:
        return arr.copy()
    pivot = arr[len(arr) // 2]
    lower = [x for x in arr if x < pivot]
    equal = [x for x in arr if x == pivot]
    upper = [x for x in arr if x > pivot]
    return quick_sort(lower) + equal + quick_sort(upper)


def heap_sort(arr):
    a = arr.copy()
    n = len(a)

    def sift_down(end, root):
        while True:
            child = 2 * root + 1
            if child >= end:
                return
            if child + 1 < end and a[child] < a[child + 1]:
                child += 1
            if a[root] >= a[child]:
                return
            a[root], a[child] = a[child], a[root]
            root = child

    for start in range(n // 2 - 1, -1, -1):
        sift_down(n, start)
    for end in range(n - 1, 0, -1):
        a[0], a[end] = a[end], a[0]
        sift_down(end, 0)
    return a


algorithms = {
    "Bubble Sort": bubble_sort,
    "Selection Sort": selection_sort,
    "Insertion Sort": insertion_sort,
    "Merge Sort": merge_sort,
    "Quick Sort": quick_sort,
    "Heap Sort": heap_sort,
}
print(f"Implemented {len(algorithms)} sorting algorithms:", ", ".join(algorithms))

## 4. Validate correctness before benchmarking

Timing a wrong implementation would make the comparison meaningless. Each algorithm is checked against Python's built-in `sorted()` using duplicates, negative values, and mixed ordering.

In [ ]:
test_data = [23, -4, 12, 12, 0, 91, 5, -4]
expected = sorted(test_data)
validation_rows = []
for name, sort_function in algorithms.items():
    actual = sort_function(test_data)
    passed = actual == expected
    validation_rows.append({"Algorithm": name, "Correct output": passed})
    if not passed:
        raise AssertionError(f"{name} failed the correctness test.")

validation_df = pd.DataFrame(validation_rows)
display(validation_df)
print("All algorithms passed the correctness test.")

## 5. Complexity reference

Theoretical complexity explains expected growth; measured timing below provides empirical evidence for the selected input and this computer.

| Algorithm | Best time | Average time | Worst time | Typical extra space |
|---|---:|---:|---:|---:|
| Bubble Sort (optimized) | O(n) | O(n²) | O(n²) | O(1) |
| Selection Sort | O(n²) | O(n²) | O(n²) | O(1) |
| Insertion Sort | O(n) | O(n²) | O(n²) | O(1) |
| Merge Sort | O(n log n) | O(n log n) | O(n log n) | O(n) |
| Quick Sort (this implementation) | O(n log n) | O(n log n) | O(n²) | O(n) in this list-based version |
| Heap Sort | O(n log n) | O(n log n) | O(n log n) | O(1) auxiliary array space |

## 6. Interactive Smart Recommendation — main project feature

1. Choose **input size** with the slider.
2. Choose the **input pattern**.
3. Click **Recommend Best Algorithm**.
4. The system runs all six algorithms several times on equivalent data, checks the sorted result, displays a timing table and chart, and names the fastest observed algorithm.

Tip: use smaller sizes for a quick demonstration. Bubble Sort and Selection Sort become slow as the input grows because they have quadratic time complexity.

In [ ]:
# Measuring helper: each sort function works on a copy of the same data.
def measure_average_time(sort_function, data, runs=3):
    if runs < 1:
        raise ValueError("runs must be at least 1")
    timings = []
    for _ in range(runs):
        start = time.perf_counter()
        sorted_result = sort_function(data)
        timings.append(time.perf_counter() - start)
        if sorted_result != sorted(data):
            raise AssertionError("An algorithm returned an incorrect result during benchmarking.")
    return sum(timings) / len(timings)

# Input controls
size_slider = widgets.IntSlider(
    value=1000, min=100, max=4000, step=100,
    description="Input size:", continuous_update=False,
    style={"description_width": "initial"},
    layout=widgets.Layout(width="420px")
)
input_dropdown = widgets.Dropdown(
    options=[
        ("Random values", "random"),
        ("Already sorted", "sorted"),
        ("Reverse sorted", "reverse"),
        ("Nearly sorted", "nearly_sorted"),
    ],
    value="random", description="Input pattern:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="420px")
)
run_button = widgets.Button(
    description="Recommend Best Algorithm",
    button_style="success", icon="check",
    tooltip="Benchmark all six algorithms and recommend the fastest observed one",
    layout=widgets.Layout(width="280px", height="42px")
)
recommendation_output = widgets.Output()


def run_recommendation(_button):
    with recommendation_output:
        clear_output(wait=True)
        size = int(size_slider.value)
        input_type = input_dropdown.value
        data = generate_data(size, input_type)
        expected = sorted(data)
        rows = []

        print("SMART SORTING ALGORITHM PERFORMANCE ANALYZER")
        print("=" * 48)
        print(f"Dataset size: {size:,} elements")
        print(f"Input pattern: {input_type.replace('_', ' ').title()}")
        print("Runs per algorithm: 3")
        print("\nBenchmarking algorithms...\n")

        for name, sort_function in algorithms.items():
            elapsed = measure_average_time(sort_function, data, runs=3)
            correct = sort_function(data) == expected
            rows.append({
                "Algorithm": name,
                "Average Time (seconds)": elapsed,
                "Average Time (milliseconds)": elapsed * 1000,
                "Correct": correct,
            })

        results_df = pd.DataFrame(rows).sort_values(
            "Average Time (seconds)", ascending=True
        ).reset_index(drop=True)
        winner = results_df.iloc[0]

        print("🏆 RECOMMENDATION FOR THIS INPUT")
        print(f"Best observed algorithm: {winner['Algorithm']}")
        print(f"Average execution time: {winner['Average Time (milliseconds)']:.4f} ms")
        print("\nAll algorithms returned correctly sorted output.")
        print("Note: this is an empirical result for the selected input and current session, not a universal winner.\n")

        display(results_df[["Algorithm", "Average Time (milliseconds)", "Correct"]].style.format({
            "Average Time (milliseconds)": "{:.4f}"
        }))

        fig, ax = plt.subplots(figsize=(10, 5))
        plot_df = results_df.sort_values("Average Time (milliseconds)", ascending=True)
        ax.bar(plot_df["Algorithm"], plot_df["Average Time (milliseconds)"])
        ax.set_title(f"Measured sorting performance — {size:,} items ({input_type.replace('_', ' ')})")
        ax.set_xlabel("Sorting algorithm")
        ax.set_ylabel("Average time (milliseconds; lower is faster)")
        ax.tick_params(axis="x", rotation=30)
        ax.grid(axis="y", alpha=0.25)
        fig.tight_layout()
        plt.show()

        # Save the most recent interactive result in the Colab working directory.
        results_df.to_csv("sorting_recommendation_results.csv", index=False)
        print("Saved detailed results to sorting_recommendation_results.csv")
        display(FileLink("sorting_recommendation_results.csv"))

run_button.on_click(run_recommendation)

controls = widgets.VBox([
    widgets.HTML("<h3 style='margin-bottom:4px'>Choose your experiment</h3>"),
    size_slider,
    input_dropdown,
    run_button,
    widgets.HTML("<small>For a fast demonstration, start with 1,000 elements. Larger inputs can take longer.</small>"),
])
display(controls)
display(recommendation_output)

## 7. How the recommendation works

- The input generator creates one dataset matching the selected size and pattern.
- Every algorithm receives that same dataset, and its function sorts a copy so one algorithm does not change another algorithm's input.
- Each algorithm is timed three times using `time.perf_counter()`; the average is used for ranking.
- The algorithm with the lowest measured average time is recommended for that experiment.
- A correctness check ensures the returned list matches Python's `sorted()` output.

**Important limitation:** micro-benchmarks are affected by the computer, Python runtime, and small timing fluctuations. Big-O complexity and the observed measurements should be considered together.

## 8. Project conclusion

This mini project combines algorithm implementation, correctness testing, experimental analysis, visualization, and a button-driven recommendation interface. It shows how input size and input order influence the measured performance of classical sorting algorithms and provides a simple evidence-based recommendation for each selected experiment.

**For the project demo:** select a small size first, try Random and Already Sorted inputs, click the recommendation button for each, then explain why the winning algorithm can differ by input pattern and measurement conditions.